In [ ]:
# MAGIC %md
# MAGIC ## Silver Layer Transformation and Target Sink Writer
# MAGIC Triggered directly by IngestionOrchestrator right after a table's S3 landing write
# MAGIC completes (see orchestrator.py `_trigger_silver`). All config values (target
# MAGIC location, format, write mode, child config table) are passed in as widgets —
# MAGIC this notebook does not re-query config_master/ingestion_config itself.

In [ ]:
from datetime import datetime
from pyspark.sql import functions as F

# ── Define Widgets ─────────────────────────────────────────────────────────
dbutils.widgets.text("config_id",       "", "Ingestion config ID")
dbutils.widgets.text("landing_path",    "", "Exact S3 landing path the Bronze write just wrote to")
dbutils.widgets.text("file_format",     "parquet", "Format landing_path was written in")
dbutils.widgets.text("target_catalog",  "", "Silver target catalog")
dbutils.widgets.text("target_schema",   "", "Silver target schema")
dbutils.widgets.text("target_table",    "", "Silver target table")
dbutils.widgets.text("write_mode",      "overwrite", "Spark write mode for the Silver write")
dbutils.widgets.text("child_table_fqn", "", "Ingestion config table this task came from (for execution-date updates)")

config_id       = int(dbutils.widgets.get("config_id"))
landing_path    = dbutils.widgets.get("landing_path")
file_format     = dbutils.widgets.get("file_format") or "parquet"
target_catalog  = dbutils.widgets.get("target_catalog")
target_schema   = dbutils.widgets.get("target_schema")
target_table    = dbutils.widgets.get("target_table")
write_mode      = dbutils.widgets.get("write_mode") or "overwrite"
child_table_fqn = dbutils.widgets.get("child_table_fqn")

target_fqn = f"{target_catalog}.{target_schema}.{target_table}"

start_time = datetime.utcnow()
print(f"[SILVER] Starting Silver process for config_id={config_id} at {start_time}")
print(f"[SILVER] Source: {landing_path} (format={file_format})")
print(f"[SILVER] Target: {target_fqn} (mode={write_mode})")

In [ ]:
# MAGIC %md
# MAGIC ### 1. Read Landing Data from S3

In [ ]:
print(f"[SILVER] Reading landing data from: {landing_path}")

df = spark.read.format(file_format).load(landing_path)
print(f"Loaded DataFrame row count: {df.count()}")

In [ ]:
# MAGIC %md
# MAGIC ### 2. Write to Silver Medallion Layer Delta Target

In [ ]:
# Ensure target schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

print(f"[SILVER] Writing Silver data to Delta table: {target_fqn}")
df.write.format("delta").mode(write_mode).option("mergeSchema", "true").saveAsTable(target_fqn)
print(f"[SILVER] Successfully wrote table: {target_fqn}")

In [ ]:
# MAGIC %md
# MAGIC ### 3. Update Configuration Execution Dates

In [0]:
end_time = datetime.utcnow()

# Resolve case-sensitive column names of the child configuration table
columns = spark.table(child_table_fqn).columns

config_id_col = next((c for c in columns if c.lower() == "config_id"), "Config_ID")
last_sink_col = next((c for c in columns if c.lower() == "silver_sink_last_sink_date"), "silver_sink_last_sink_date")
started_col   = next((c for c in columns if c.lower() == "sink_batch_started_date"), "sink_batch_started_date")

print(f"Updating metadata dates in child table: {child_table_fqn}")
print(f"  Start: {start_time}")
print(f"  End: {end_time}")

spark.sql(f"""
    UPDATE {child_table_fqn}
    SET {last_sink_col} = '{end_time}',
        {started_col}   = '{start_time}'
    WHERE {config_id_col} = {config_id}
""")
print(f"[SILVER] Metadata update completed successfully.")